# CORAL Tutorial — Multi-scale Multi-modal Integration of Spatial Omics

*March 2026*

Siyu He

## Overview

**CORAL** is a deep generative model that integrates spatial omics data across multiple modalities and resolutions. CORAL combines high-resolution spatial protein data (e.g., CODEX) with lower-resolution spatial transcriptomics data (e.g., Visium) to produce a unified latent representation and downstream analysis.

### What this tutorial covers

1. Loading and visualizing multi-modal spatial omics data
2. Preparing data for CORAL (preprocessing and graph construction)
3. Training the CORAL model
4. Running inference to obtain integrated embeddings
5. Evaluating CORAL embeddings against ground truth annotations
6. Generating enriched low-resolution expression profiles

### Dataset

We use a mouse thymus dataset with:
- **High-resolution (CODEX):** 4,697 cells × 51 proteins
- **Low-resolution (Visium):** ~200 spots × 3,036 genes
- **Ground truth:** Manual cell-type annotations

### Runtime

This tutorial takes approximately **15 minutes** on a GPU-equipped machine.

## Prerequisites

**Installation:**

```bash
pip install git+https://github.com/zou-group/CORAL
```

Or install from source:

```bash
git clone https://github.com/shsiyu/CORAL.git
cd CORAL
pip install -e .
```

**Dependencies:** scanpy, anndata, torch, torch_geometric, scikit-learn, umap-learn, scipy, matplotlib, seaborn

**Hardware:** GPU recommended (NVIDIA GPU with CUDA support). CPU execution is supported but significantly slower.

## Step 1: Import Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import scanpy as sc
import anndata
import torch

from sklearn.metrics import adjusted_rand_score

In [ ]:
import coral

## Step 2: Download and Load Data

Download the mouse thymus dataset from Figshare (doi: [10.6084/m9.figshare.30676556](https://doi.org/10.6084/m9.figshare.30676556)). The download is skipped automatically if the files already exist.

In [ ]:
import urllib.request
import os

data_dir = "Mouse_thymus_data"
os.makedirs(data_dir, exist_ok=True)

files = {
    "adata_thymus1_annotation.h5ad": "https://ndownloader.figshare.com/files/59752970",
    "adata_ADT.h5ad": "https://ndownloader.figshare.com/files/59752967",
    "adata_RNA_low.h5ad": "https://ndownloader.figshare.com/files/59752973",
}

for filename, url in files.items():
    filepath = os.path.join(data_dir, filename)
    if os.path.exists(filepath):
        print(f"Already exists: {filepath}")
    else:
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filepath)
        print(f"  Saved to {filepath}")

In [ ]:
# Load the three h5ad files
ground_truth_adata = sc.read_h5ad('Mouse_thymus_data/adata_thymus1_annotation.h5ad')
hires_adata = sc.read_h5ad('Mouse_thymus_data/adata_ADT.h5ad')
lowres_adata = sc.read_h5ad('Mouse_thymus_data/adata_RNA_low.h5ad')

hires_adata = hires_adata[hires_adata.obs_names, :]
hires_adata.obs['Annotation'] = ground_truth_adata.obs['Annotation'].astype(str)

lowres_adata = lowres_adata[lowres_adata.obs_names, :]

## Step 3: (Optional) Visualize Input Data

Before running CORAL, inspect the spatial organization of both modalities. The high-resolution data (CODEX) captures protein expression at single-cell resolution, while the low-resolution data (Visium) measures transcriptomes at a larger spatial scale.

Leiden clustering is used for initial unsupervised visualization. Adjust the `res` parameter to control cluster granularity.

In [ ]:
coral.utils.plot_spatial(
    hires_adata,
    res=0.7,
    use_rep_for_cluster='X_pca',
    to_plot_var='cluster',
    need_lognormed=True,
    size=1,
    figsize=(3.5, 3),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])

In [ ]:
coral.utils.plot_spatial(
    lowres_adata,
    res=1.6235,
    use_rep_for_cluster='X_pca',
    to_plot_var='cluster',
    need_lognormed=True,
    size=10,
    figsize=(3.5, 2.7),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    legend_fontsize=10,
    legend_markerscale=2,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])

## Step 4: (Optional) Visualize Ground Truth Annotations

The ground truth annotations provide manually curated cell-type labels for the high-resolution data. We visualize these in spatial coordinates and as a UMAP embedding to understand the expected tissue organization.

In [ ]:
coral.utils.plot_spatial(
    ground_truth_adata,
    res=0.7,
    use_rep_for_cluster=None,
    to_plot_var='Annotation',
    need_lognormed=False,
    size=1,
    figsize=(3.5, 3),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])

In [ ]:
coral.utils.plot_umap(
    hires_adata,
    res=0.7,
    use_rep_for_cluster='X_pca',
    to_plot_var='Annotation',
    need_lognormed=True,
    size=15,
    figsize=(10, 5),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])

In [ ]:
coral.utils.plot_umap_gene(
    hires_adata,
    res=0.7,
    use_rep_for_cluster='X_pca',
    to_plot_gene='Mouse-CD19',
    need_lognormed=True,
    size=10,
    figsize=(3, 3),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    vmin=0,
    vmax=6)

## Step 5: Assign Cell Types to High-Resolution Data

CORAL requires cell-type labels for the high-resolution data, used as conditioning variables during training. Here we use Leiden clustering as a proxy for cell types.

**Key parameters:**
- `res`: Leiden clustering resolution. Higher values produce more clusters. Adjust based on expected number of cell types in your data.
- `use_rep_for_cluster`: Representation used for computing the neighbor graph (e.g., `'X_pca'`).

> **Note:** If manual annotations are available, use those instead of unsupervised clusters for better results.

In [ ]:
hires_adata = coral.utils.add_cluster(
    hires_adata,
    res=0.7,
    use_rep_for_cluster='X_pca',
    need_lognormed=True)

hires_adata.obs['cell_type'] = hires_adata.obs['cluster']

## Step 6: Prepare Data for CORAL

This step preprocesses the input data and constructs local spatial subgraphs for training:

1. **`preprocess_data`** aligns high-resolution cells to their nearest low-resolution spots, concatenates expression matrices, and computes one-hot encoded cell-type vectors.
2. **`prepare_local_subgraphs`** builds k-nearest-neighbor spatial graphs where each subgraph is centered on a high-resolution cell and its local neighborhood.

**Key parameters:**
- `n_neighbors`: Number of spatial neighbors for graph construction (default: 40). Increase for denser tissues; decrease for sparser ones.

In [ ]:
combined_expr, hires_coords, one_hot_cell_types, spot_indices, lowres_expr = coral.utils.preprocess_data(
    lowres_adata, hires_adata)

dataloader = coral.utils.prepare_local_subgraphs(
    combined_expr, hires_coords, one_hot_cell_types,
    spot_indices, lowres_expr, n_neighbors=40)

## Step 7: Create CORAL Model

Initialize the CORAL model architecture. The model consists of:

- **Separate encoders** for high-resolution (protein) and low-resolution (RNA) modalities
- **Graph Attention Network (GAT)** layers for spatial context encoding
- **Cross-attention** between modalities
- **Variational inference** with latent variables *z* (shared embedding) and *v* (nuisance)
- **Deconvolution layer** for cell-type aware expression reconstruction

**Key parameters:**
- `latent_dim`: Dimension of the shared latent space (default: 64). Larger values capture more variation.
- `hidden_channels`: Hidden layer width in GAT (default: 128).
- `v_dim`: Dimension of the nuisance variable *v* (default: 1).

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    print("GPU is available.")
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs available: {gpu_count}")
    for i in range(gpu_count):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("GPU is not available. Training will be slower.")

In [ ]:
model, optimizer = coral.model.create_model(
    lowres_dim=lowres_adata.shape[1],
    hires_dim=hires_adata.shape[1],
    lowres_size=lowres_adata.shape[0],
    hires_size=hires_adata.shape[0],
    cell_type_dim=one_hot_cell_types.shape[1],
    latent_dim=64,
    hidden_channels=128,
    v_dim=1)

model.to(device)

## Step 8: Train the Model

Train the CORAL model using the prepared dataloader. The loss function includes:
- Reconstruction losses (Negative Binomial for RNA, Gamma for protein)
- KL divergence for latent variables *z* and *v*
- Contrastive loss for cross-modal alignment
- Graph Laplacian regularization for spatial smoothness

**Key parameters:**
- `epochs`: Number of training epochs (100 is sufficient for this dataset). Larger or more complex datasets may need 200–300 epochs.

Training progress is printed as per-epoch loss values. Expect the loss to decrease and stabilize.

In [ ]:
coral.trainer.train_model(model, optimizer, dataloader, epochs=100, device=device)

In [ ]:
model_save_path = "model.pth"
optimizer_save_path = "optimizer.pth"

torch.save(model.state_dict(), model_save_path)
torch.save(optimizer.state_dict(), optimizer_save_path)
print(f"Model saved to {model_save_path}")

## Step 9: (Optional) Load Pre-trained Model

If you have previously trained a model, you can load it here instead of retraining.

In [ ]:
model_save_path = "model.pth"
optimizer_save_path = "optimizer.pth"

model.load_state_dict(torch.load(model_save_path))
optimizer.load_state_dict(torch.load(optimizer_save_path))
print("Model loaded successfully.")

## Step 10: Run Inference

Run the trained model on all subgraphs to generate:

- **CORAL embeddings** (`adata.obsm['coral']`): Unified latent representation integrating both modalities
- **Generated expression** (`adata.obsm['generated_expr']`): Deconvolved low-resolution RNA expression at single-cell resolution
- **Edge indices and attention weights**: Graph structure and learned spatial attention for downstream analysis

The output AnnData object is reindexed to match the original high-resolution data ordering.

In [ ]:
adata_model_gener, edges_all, attn_weights_all = coral.inference.generate_and_validate(
    model, dataloader, device, hires_adata)

adata_model_gener

## Step 11: Analyze CORAL Embeddings

Evaluate the quality of CORAL's learned embeddings by:

- **Spatial cluster plot**: Leiden clustering on CORAL embeddings, plotted in tissue coordinates
- **UMAP visualization**: CORAL embedding UMAP colored by ground truth annotations
- **Adjusted Rand Index (ARI)**: Quantitative measure of clustering agreement with ground truth (1.0 = perfect agreement)

In [ ]:
# Spatial clusters from CORAL embeddings
coral.utils.plot_spatial(
    adata_model_gener,
    res=0.82,
    use_rep_for_cluster='coral',
    to_plot_var='cluster',
    need_lognormed=True,
    size=5,
    figsize=(4.5, 4.2),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])

In [ ]:
# Add ground truth annotations to the inference output
adata_model_gener.obs['Annotation'] = hires_adata.obs['Annotation'].values

# UMAP of CORAL latent embeddings colored by ground truth annotations
coral.utils.plot_latent_umap(
    adata_model_gener,
    rep='coral',
    to_plot_var='Annotation',
    custom_palette=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                    '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                    '#affc41', 'k', 'y', 'g'])

In [ ]:
# Compute CORAL embedding clusters using Leiden
adata_eval = adata_model_gener.copy()
sc.pp.neighbors(adata_eval, n_neighbors=100, use_rep='coral')
sc.tl.leiden(adata_eval, resolution=0.82, random_state=0, flavor='igraph')

# Adjusted Rand Index: CORAL clusters vs ground truth annotations
ari = adjusted_rand_score(adata_eval.obs['Annotation'], adata_eval.obs['leiden'])
print(f"Adjusted Rand Index (CORAL clusters vs ground truth): {ari:.4f}")

## Step 12: Enriched Low-Resolution Data

CORAL produces deconvolved gene expression profiles at single-cell resolution from the low-resolution modality. The `generated_expr` field in `adata_model_gener.obsm` contains these enriched profiles — effectively "super-resolving" the Visium spots into individual cell-level RNA expression estimates.

Below we create a standalone AnnData from these enriched profiles and visualize its spatial clusters.

In [ ]:
# Create AnnData from deconvolved expression
enriched_lowres_adata = anndata.AnnData(adata_model_gener.obsm['generated_expr'])
enriched_lowres_adata.obsm = adata_model_gener.obsm.copy()

# Add gene names from the original low-resolution data if dimensions match
if enriched_lowres_adata.shape[1] <= lowres_adata.shape[1]:
    enriched_lowres_adata.var_names = lowres_adata.var_names[:enriched_lowres_adata.shape[1]].copy()

print(f"Enriched low-resolution AnnData: {enriched_lowres_adata.shape[0]} cells x {enriched_lowres_adata.shape[1]} genes")

In [ ]:
coral.utils.plot_spatial(
    enriched_lowres_adata,
    res=1.3,
    use_rep_for_cluster='X_pca',
    to_plot_var='cluster',
    need_lognormed=True,
    size=5,
    figsize=(4.5, 4.2),
    legd=True,
    invert_yaxis=True,
    axis_=False,
    legend_fontsize=10,
    legend_markerscale=2,
    color_list=['#9f86c0', '#d4e09b', '#ff9f1c', '#fdc500', '#00509d', '#8ecae6',
                '#dc2f02', '#00296b', '#219ebc', '#126782', '#023047', '#ffc9b9',
                '#affc41', 'k', 'y', 'g'])